# The notebook as an instrument (advanced version)
This notebook works like a measuring instrument, but this time you also build a few of its parts. In the simple version you only turned the knobs; here you write short pieces of code using pandas, numpy and matplotlib.

**Rules:**
1. Run the cells from top to bottom (Shift + Enter).
2. Only edit two kinds of places: the boxes marked "YOU MAY CHANGE THIS" (values such as file names, column names and times) and the places marked "YOUR CODE HERE" (lines you complete yourself).
3. Each cell ends with a check. If it prints OK, you are fine. If it stops with an error, read the message: it tells you what is missing.
4. Each cell has a hint with the names of the functions you need. Look them up in the documentation, or use prompts 4 to 6 on the LLM card. Ask for hints, not for the full solution: the worksheet asks you to explain your own code.
5. If something comes out in red, read the last line of the message.

Do the simple version first, on the reference CSV.

## Setup
Loads the libraries and defines `check()`, a small function that tells you whether your code does what the cell expects. Nothing to change here, just run it.

In [ ]:
# ==============================================================
#  SETUP
#  What it does: loads the libraries and defines check().
#  Nothing to change here.
# ==============================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def check(condition, ok_message, error_message):
    # Prints OK if the condition is true; otherwise stops with an error.
    if condition:
        print("OK:", ok_message)
    else:
        raise AssertionError("CHECK FAILED: " + error_message)

## Cell 1 — Load and inspect the data
This cell opens your CSV file and you complete the code that finds three basic facts about the recording: which signals it contains, how often it was sampled, and how long it lasts.

Hint: `pd.read_csv`, `data.columns`, a list comprehension (`[c for c in ... if ...]`), `.diff()`, `.median()`, `.iloc[-1]`. The sampling frequency is 1 divided by the time between two consecutive samples.

In [ ]:
# ==============================================================
#  CELL 1 — LOAD AND INSPECT THE DATA
#  What it does: opens a CSV file and computes the signal list,
#  the sampling frequency and the duration of the recording.
# ==============================================================

# --------------------------------------------------------------
#  >>> YOU MAY CHANGE THIS <<<
#  FILE_NAME:   the name of your file, in quotes.
#  TIME_COLUMN: the EXACT name of the time column (in seconds).
#  Typical error: FileNotFoundError -> the file name is wrong or
#  the file is not uploaded. KeyError -> TIME_COLUMN does not
#  exist in your file; copy it from the column list below.
# --------------------------------------------------------------
FILE_NAME = "reference_data.csv"
TIME_COLUMN = "time_s"
# --------------------------------------------------------------

# --------------------------------------------------------------
#  >>> YOUR CODE HERE <<<
#  (1a) Load the CSV into a pandas table called data.
#  (1b) signal_columns: a list with all column names except the
#       time column.
#  (1c) fs: sampling frequency in Hz, equal to 1 divided by the
#       median time step between consecutive samples.
#  (1d) duration: length of the recording in seconds (last time
#       minus first time).
#  Replace each None with your code.
# --------------------------------------------------------------
data = None             # replace None (1a)
signal_columns = None   # replace None (1b)
fs = None               # replace None (1c)
duration = None         # replace None (1d)
# --------------------------------------------------------------

# --- From here on do NOT change anything ---
check(isinstance(data, pd.DataFrame), "data is a pandas table.",
      "data is not a table yet. Complete (1a) with pd.read_csv(...).")
check(TIME_COLUMN in data.columns, f"time column '{TIME_COLUMN}' found.",
      f"there is no column '{TIME_COLUMN}'. Available columns: {list(data.columns)}")
check(isinstance(signal_columns, list) and len(signal_columns) > 0 and TIME_COLUMN not in signal_columns,
      f"{len(signal_columns)} signals found.",
      "signal_columns must be a list of all columns except the time column (1b).")
check(fs is not None and fs > 0, f"sampling frequency is {fs:.1f} Hz.", "(1c) fs must be a positive number.")
check(duration is not None and duration > 0, f"duration is {duration:.1f} s.", "(1d) duration must be a positive number.")

print()
print("First rows:")
print(data.head())
print()
print("Data type of each column:")
print(data.dtypes)
print()
print("Missing values (NaN) per column:")
print(data.isna().sum())

## Cell 2 — Plot all the signals
This cell draws every signal against time. You write the loop that does the drawing, and you fill in a dictionary with the label and unit of each signal (the ones you defined in Session 3).

Hint: `for ax, col in zip(axes, signal_columns):`, `ax.plot(x, y)`, `ax.set_ylabel(text)`, `LABELS.get(col, col)` (returns the label, or the column name if there is none).

In [ ]:
# ==============================================================
#  CELL 2 — PLOT ALL THE SIGNALS
#  What it does: draws one subplot per signal, all sharing the
#  same time axis.
# ==============================================================

# --------------------------------------------------------------
#  >>> YOU MAY CHANGE THIS <<<
#  LABELS: a dictionary  "column name": "Quantity (unit)".
#  Use your labels from Session 3. Example:
#  "signal_1": "Heart rate (bpm)"
#  Typical error: SyntaxError -> a comma or a quote is missing.
# --------------------------------------------------------------
LABELS = {
    "signal_1": "Signal 1 (unit)",
    "signal_2": "Signal 2 (unit)",
    "signal_3": "Signal 3 (unit)",
    "signal_4": "Signal 4 (unit)",
    "signal_5": "Signal 5 (unit)",
    "signal_6": "Signal 6 (unit)",
}
# --------------------------------------------------------------

fig, axes = plt.subplots(len(signal_columns), 1, sharex=True,
                         figsize=(10, 2 * len(signal_columns)))
axes = np.atleast_1d(axes)   # axes is always a list, even with one signal

# --------------------------------------------------------------
#  >>> YOUR CODE HERE <<<
#  Write a for loop over (ax, col). In each subplot, plot the
#  signal against time and set its y-axis label from LABELS.
# --------------------------------------------------------------
# <TODO 2>
# --------------------------------------------------------------

# --- From here on do NOT change anything ---
axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
check(all(len(ax.lines) == 1 for ax in axes), "every subplot contains its signal.",
      "some subplots are empty or have more than one line. Check your loop.")
check(all(ax.get_ylabel() != "" for ax in axes), "every y-axis has a label.",
      "a y-axis has no label. Use ax.set_ylabel(...) inside the loop.")
plt.show()

## Cell 3 — Look at one signal in detail
Choose ONE signal. You compute its statistics and smooth it with a moving average, then compare the original with the smoothed version to estimate how noisy it is.

Hint: `data[signal_columns].agg([...])`, `data[col].rolling(window=N, center=True).mean()`, `.std()`. A moving average replaces each point by the mean of its neighbours; N samples correspond to N / fs seconds.

In [ ]:
# ==============================================================
#  CELL 3 — ONE SIGNAL: STATISTICS AND SMOOTHING
#  What it does: prints summary statistics, smooths one signal
#  and plots the original and the smoothed version together.
# ==============================================================

# --------------------------------------------------------------
#  >>> YOU MAY CHANGE THIS <<<
#  COLUMN_NAME:   the EXACT name of the column (see the list in
#                 Cell 1), in quotes.
#  SMOOTH_WINDOW: number of samples in the moving average (a whole
#                 number). Bigger = smoother, but less detail.
#  Typical error: KeyError -> the column name does not exist as
#  you typed it (capital letters? spaces?).
# --------------------------------------------------------------
COLUMN_NAME = "signal_1"
SMOOTH_WINDOW = 25
# --------------------------------------------------------------
Y_LABEL = LABELS.get(COLUMN_NAME, COLUMN_NAME)

# --------------------------------------------------------------
#  >>> YOUR CODE HERE <<<
#  (3a) stats: a table with the mean, std, min and max (in this
#       order) of ALL the signals.
#  (3b) smoothed: centred moving average of COLUMN_NAME.
#  (3c) noise: standard deviation of (original - smoothed).
#  Replace each None with your code.
# --------------------------------------------------------------
stats = None      # replace None (3a)
smoothed = None   # replace None (3b)
noise = None      # replace None (3c)
# --------------------------------------------------------------

# --- From here on do NOT change anything ---
check(isinstance(stats, pd.DataFrame) and list(stats.index) == ["mean", "std", "min", "max"],
      "statistics table is correct.", "(3a) stats must have the rows mean, std, min, max, in this order.")
check(isinstance(smoothed, pd.Series) and len(smoothed) == len(data),
      "smoothed signal is correct.", "(3b) smoothed must be a series as long as data.")
check(noise is not None and noise >= 0, f"estimated noise is {noise:.3f}.", "(3c) noise must be a number >= 0.")

print()
print(stats.round(2))
print()
print(f"Smoothing window: {SMOOTH_WINDOW} samples = {SMOOTH_WINDOW / fs:.2f} s")

plt.figure(figsize=(10, 4))
plt.plot(data[TIME_COLUMN], data[COLUMN_NAME], alpha=0.4, label="original")
plt.plot(data[TIME_COLUMN], smoothed, label=f"moving average ({SMOOTH_WINDOW} samples)")
plt.xlabel("Time (s)")
plt.ylabel(Y_LABEL)
plt.legend()
plt.show()

## Cell 4 — Write a zoom function
Instead of editing times by hand every time, you write one reusable function, `plot_window`, that cuts a time window out of a signal and plots it with a title. In Cell 5 you will use it on your five movements. This cell only defines the function, so it draws nothing yet.

Hint: a boolean mask looks like `df[(df[col] >= a) & (df[col] <= b)]`; `window.empty` is True when no rows are left; `raise ValueError("message")` stops the function with a clear error; `ax.axvline(t, linestyle="--")` draws a vertical line at time t.

In [ ]:
# ==============================================================
#  CELL 4 — THE FUNCTION plot_window
#  What it does: defines a function that plots one signal between
#  t_start and t_end, with title, axis labels and optional vertical
#  lines at the tap markers. It returns the figure.
#  Nothing is drawn until you call it in Cell 5.
# ==============================================================
def plot_window(df, column, t_start, t_end, title, y_label, markers=()):

    # ----------------------------------------------------------
    #  >>> YOUR CODE HERE <<<
    #  (4a) window: the rows of df whose time is between t_start
    #       and t_end (both included).
    # ----------------------------------------------------------
    window = None   # replace None (4a)

    # ----------------------------------------------------------
    #  (4b) If window is empty, raise a ValueError that says the
    #       interval is outside the range of the data.
    # ----------------------------------------------------------
    # <TODO 4b>

    fig, ax = plt.subplots(figsize=(10, 4))

    # ----------------------------------------------------------
    #  (4c) Plot column of window against time.
    # ----------------------------------------------------------
    # <TODO 4c>

    ax.set_xlabel("Time (s)")
    ax.set_ylabel(y_label)
    ax.set_title(title)

    # ----------------------------------------------------------
    #  (4d) For each time t in markers that lies inside the window,
    #       draw a dashed vertical line.
    # ----------------------------------------------------------
    # <TODO 4d>

    return fig

## Cell 5 — Zoom to the five movements
Use your function on each of the five movements from Session 3. You describe each movement in a list and write the loop that calls the function.

Hint: a loop over a list of dictionaries: `for m in MOVEMENTS:` and then `m["start"]`, `m["end"]`, and so on. Store each figure in the dictionary `figures` using its name as key: `figures[name] = fig`.

In [ ]:
# ==============================================================
#  CELL 5 — ZOOM TO THE FIVE MOVEMENTS
#  What it does: calls plot_window once for each movement.
# ==============================================================

# --------------------------------------------------------------
#  >>> YOU MAY CHANGE THIS <<<
#  One dictionary per movement:
#  name:   name of the picture file, without .png and without spaces
#  column: the EXACT name of the signal that shows this movement best
#  start:  second where the cut begins (a number)
#  end:    second where it ends (must be LARGER than start)
#  title:  title of the plot; say what the movement is
#  taps:   times (in s) of the tap markers, for example [6.2, 9.8];
#          leave [] if there are none
#  Typical error: ValueError "Empty window" -> your interval is
#  outside the range of your data. Look at the time axis in Cell 2.
# --------------------------------------------------------------
MOVEMENTS = [
    {"name": "movement_1", "column": "signal_1", "start": 5, "end": 15, "title": "Movement 1: describe it here", "taps": []},
    {"name": "movement_2", "column": "signal_1", "start": 5, "end": 15, "title": "Movement 2: describe it here", "taps": []},
    {"name": "movement_3", "column": "signal_1", "start": 5, "end": 15, "title": "Movement 3: describe it here", "taps": []},
    {"name": "movement_4", "column": "signal_1", "start": 5, "end": 15, "title": "Movement 4: describe it here", "taps": []},
    {"name": "movement_5", "column": "signal_1", "start": 5, "end": 15, "title": "Movement 5: describe it here", "taps": []},
]
# --------------------------------------------------------------

figures = {}   # each figure will be stored here under its name

# --------------------------------------------------------------
#  >>> YOUR CODE HERE <<<
#  Loop over MOVEMENTS. For each movement m, call plot_window with
#  its values (use LABELS.get(m["column"], m["column"]) as y_label)
#  and store the figure in figures[m["name"]].
# --------------------------------------------------------------
# <TODO 5>
# --------------------------------------------------------------

# --- From here on do NOT change anything ---
check(len(figures) == len(MOVEMENTS), f"{len(figures)} figures created.",
      "there must be one figure per movement in figures.")
plt.show()

## Cell 6 — Save your figures
Write the loop that saves every figure as a PNG image. Find the files in the file panel (left), right click, Download.

Hint: `figures.items()` gives pairs `(name, figure)`; `fig.savefig(file_name, dpi=150, bbox_inches="tight")`; an f-string such as `f"{name}.png"` builds the file name.

In [ ]:
# ==============================================================
#  CELL 6 — SAVE THE FIGURES
#  What it does: saves each figure from Cell 5 as a PNG image.
# ==============================================================

# --------------------------------------------------------------
#  >>> YOUR CODE HERE <<<
#  Loop over figures and save each one as "<name>.png".
# --------------------------------------------------------------
# <TODO 6>
# --------------------------------------------------------------

# --- From here on do NOT change anything ---
missing = [m["name"] + ".png" for m in MOVEMENTS if not os.path.exists(m["name"] + ".png")]
check(len(missing) == 0, "all figures are saved as PNG files.", f"these files are missing: {missing}")

---
## For lecturers only: reference solutions (delete this cell before handing out the notebook)

data = None             # replace None (1a)
```python
data = pd.read_csv(FILE_NAME)
```

signal_columns = None   # replace None (1b)
```python
signal_columns = [c for c in data.columns if c != TIME_COLUMN]
```

fs = None               # replace None (1c)
```python
fs = 1 / data[TIME_COLUMN].diff().median()
```

duration = None         # replace None (1d)
```python
duration = data[TIME_COLUMN].iloc[-1] - data[TIME_COLUMN].iloc[0]
```

# <TODO 2>
```python
for ax, col in zip(axes, signal_columns):
    ax.plot(data[TIME_COLUMN], data[col])
    ax.set_ylabel(LABELS.get(col, col))
```

stats = None      # replace None (3a)
```python
stats = data[signal_columns].agg(["mean", "std", "min", "max"])
```

smoothed = None   # replace None (3b)
```python
smoothed = data[COLUMN_NAME].rolling(window=SMOOTH_WINDOW, center=True).mean()
```

noise = None      # replace None (3c)
```python
noise = (data[COLUMN_NAME] - smoothed).std()
```

window = None   # replace None (4a)
```python
window = df[(df[TIME_COLUMN] >= t_start) & (df[TIME_COLUMN] <= t_end)]
```

# <TODO 4b>
```python
if window.empty:
        raise ValueError(f"Empty window: {t_start}-{t_end} s is outside the range of the data.")
```

# <TODO 4c>
```python
ax.plot(window[TIME_COLUMN], window[column])
```

# <TODO 4d>
```python
for t in markers:
        if t_start <= t <= t_end:
            ax.axvline(t, color="gray", linestyle="--", alpha=0.7)
```

# <TODO 5>
```python
for m in MOVEMENTS:
    figures[m["name"]] = plot_window(data, m["column"], m["start"], m["end"],
                                     m["title"], LABELS.get(m["column"], m["column"]), m["taps"])
```

# <TODO 6>
```python
for name, fig in figures.items():
    fig.savefig(f"{name}.png", dpi=150, bbox_inches="tight")
```

# <TODO 7>
```python
for m in MOVEMENTS:
    w = data[(data[TIME_COLUMN] >= m["start"]) & (data[TIME_COLUMN] <= m["end"])]
    idx = w[m["column"]].idxmax()
    rows.append({"movement": m["name"], "signal": m["column"],
                 "peak_value": w.loc[idx, m["column"]], "peak_time_s": w.loc[idx, TIME_COLUMN]})
```